<a href="https://colab.research.google.com/github/mariyagrechanaya/4353455/blob/main/GP2_Final_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Групповой проект №2: Scraping & API

## Тема: Анализ рынка аренды квартир в Москве

### Бизнес-задача
Разработать систему оценки доступности квартир для аренды в Москве на основе их расположения. Это поможет:
- Арендаторам быстрее находить удобные варианты
- Владельцам правильно оценивать стоимость
- Агентствам лучше понимать рынок

### Описание работы
1. **Сбор данных** - парсинг объявлений с сайта Циан (Web Scraping)
2. **Обогащение данных** - получение координат и информации об инфраструктуре через API
3. **Анализ** - расчет метрик доступности и исследование данных

## 1. Подготовка

### Устанавливаем библиотеки

In [ ]:
from datetime import datetime
import time
import re
import math
import pandas as pd
import requests
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

## 2. Web Scraping - Парсинг Циан

Собираем объявления об аренде квартир в Москве

In [ ]:
# Функция для очистки цены
def get_price(text):
    numbers = re.findall(r'\d+', text.replace(' ', ''))
    if numbers:
        return int(numbers[0])
    return None

# Функция для получения площади
def get_area(text):
    match = re.search(r'(\d+[.,]?\d*)\s*м', text)
    if match:
        return float(match.group(1).replace(',', '.'))
    return None

# Функция для получения комнат
def get_rooms(text):
    if 'студия' in text.lower():
        return 0
    match = re.search(r'(\d+)-комн', text)
    if match:
        return int(match.group(1))
    return None

# Функция для получения этажа
def get_floors(text):
    match = re.search(r'(\d+)/(\d+)\s*этаж', text)
    if match:
        return int(match.group(1)), int(match.group(2))
    return None, None

# Функция для обработки редирект-ссылок
def clean_url(url):
    """Извлекает финальный URL если это редирект"""
    if not url:
        return None

    # Если это редирект через go.php или similar
    if 'go.php' in url or '/go/' in url:
        # Ищем параметр url
        match = re.search(r'[?&]url=([^&]+)', url)
        if match:
            return match.group(1)

    return url

# Функция для разбора адреса на части
def parse_address_parts(labels):
    """Извлекает АО, район, улицу и дом из списка меток"""
    ao_codes = {'ЦАО', 'САО', 'СВАО', 'ВАО', 'ЮВАО', 'ЮАО', 'ЮЗАО', 'ЗАО', 'СЗАО', 'ЗелАО', 'НАО', 'ТАО'}

    ao = None
    district = None
    street = None
    house = None

    for label in labels:
        text = label.strip()
        if not text or text == 'Москва' or text.startswith('м.'):
            continue

        # Административный округ
        if text in ao_codes:
            ao = text
            continue

        # Район
        if 'р-н' in text:
            district = text.replace('р-н', '').strip()
            continue

        # Номер дома (начинается с цифры)
        if text and text[0].isdigit():
            house = text
            continue

        # Улица (содержит типичные слова)
        if any(word in text.lower() for word in [
            'улица', 'проспект', 'шоссе', 'бульвар', 'переулок', 'набережная',
            'площадь', 'аллея', 'микрорайон', 'проезд', 'тракт'
        ]):
            street = text
            continue

    return ao, district, street, house

# Функция для определения типа недвижимости
def get_property_type(text):
    """Определяет тип недвижимости из текста"""
    if not text:
        return None

    text_lower = text.lower()

    if 'апартамент' in text_lower:
        return 'апартаменты'
    if 'комната' in text_lower:
        return 'комната'
    if 'студия' in text_lower:
        return 'студия'
    if 'дом' in text_lower and 'жк' not in text_lower:
        return 'дом'
    if 'квартира' in text_lower or 'к' in text_lower:
        return 'квартира'

    return 'квартира'  # По умолчанию

# Функция для парсинга условий оплаты
def parse_payment_info(text):
    """Извлекает информацию об оплате из текста"""
    if not text:
        return None, None, None, None, None, None, None, None

    text_lower = text.lower()

    # Коммунальные услуги
    utilities_included = 'комм. платежи включены' in text_lower or 'коммунальные платежи включены' in text_lower

    # Счетчики
    counters_included = 'счётчики включены' in text_lower or 'счетчики включены' in text_lower

    # Комиссия
    commission_amount = None
    if 'без комиссии' in text_lower:
        commission_amount = 0
    else:
        commission_match = re.search(r'комисс[ияи]\s*(\d+[\s\d]*)', text_lower)
        if commission_match:
            commission_amount = int(re.sub(r'\D', '', commission_match.group(1)))

    # Залог
    deposit_amount = None
    deposit_match = re.search(r'залог\s*(\d+[\s\d]*)', text_lower)
    if deposit_match:
        deposit_amount = int(re.sub(r'\D', '', deposit_match.group(1)))

    # Минимальный срок аренды
    min_term = None
    min_term_months = None

    if 'от года' in text_lower or 'от 1 года' in text_lower:
        min_term = 'от года'
        min_term_months = 12
    else:
        months_match = re.search(r'от\s*(\d+)\s*(?:месяц|месяцев|мес\.?|месяца)', text_lower)
        if months_match:
            months = int(months_match.group(1))
            min_term = f"от {months} месяцев"
            min_term_months = months
        elif 'от месяца' in text_lower or 'от 1 месяца' in text_lower:
            min_term = 'от месяца'
            min_term_months = 1

    # Тип аренды
    is_long_term = 'на длительный срок' in text_lower or 'длительный срок' in text_lower
    is_short_term = 'посуточно' in text_lower or 'краткосроч' in text_lower

    return (utilities_included, counters_included, commission_amount, deposit_amount,
            min_term, min_term_months, is_long_term, is_short_term)

print("Функции парсинга готовы")

Функции парсинга готовы


In [ ]:
# Запускаем браузер для парсинга
options = Options()
options.add_argument('--headless')  # Без окна браузера
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)

print("Браузер запущен")

# Собираем данные с разных ценовых диапазонов
# Это нужно чтобы обойти ограничение Циан на количество страниц

all_listings = []

# Диапазоны цен для сбора (руб/мес)
price_ranges = []

price_ranges.append((40000,50000))

# # От 0 до 40000 с шагом 5000
# for price in range(5000, 41000, 5000):
#     price_ranges.append((price - 5000 if price > 5000 else None, price))

# # От 40000 до 200000 с шагом 1000
# for price in range(41000, 201000, 1000):
#     price_ranges.append((price - 1000, price))

# # От 200000 до 500000 с шагом 20000
# for price in range(220000, 501000, 20000):
#     price_ranges.append((price - 20000, price))

# # Выше 500000
# price_ranges.append((500000, None))

print(f"Будем собирать данные по {len(price_ranges)} ценовым диапазонам")

# ВАЖНО: Этот код собирает ~20000 объявлений, работает несколько часов!
# Для теста можно уменьшить max_pages или количество диапазонов

max_pages = 1  # Максимум страниц на диапазон
seen_ids = set()  # Чтобы не было дубликатов
scraped_at = datetime.now().isoformat()  # Время сбора данных

for i, (min_price, max_price) in enumerate(price_ranges):
    print(f"\nДиапазон {i+1}/{len(price_ranges)}: {min_price} - {max_price}")

    for page in range(1, max_pages + 1):
        # Формируем URL
        url = "https://www.cian.ru/cat.php?deal_type=rent&offer_type=flat&region=1&engine_version=2&type=4"

        if min_price:
            url += f"&minprice={min_price}"
        if max_price:
            url += f"&maxprice={max_price}"
        if page > 1:
            url += f"&p={page}"

        # Загружаем страницу
        driver.get(url)
        time.sleep(2)

        # Парсим HTML
        soup = BeautifulSoup(driver.page_source, 'html.parser')

        # Ищем карточки объявлений
        cards = soup.find_all('article', attrs={'data-name': 'CardComponent'})

        if not cards:
            print(f"Страница {page}: объявлений не найдено, переходим к следующему диапазону")
            break

        new_count = 0
        for card in cards:
            # Ссылка на объявление
            link = card.find('a', href=lambda h: h and '/rent/flat/' in str(h))
            if not link:
                continue

            url_full = link['href']

            # Обработка редиректов
            url_full = clean_url(url_full)
            if not url_full:
                continue

            if not url_full.startswith('http'):
                url_full = 'https://www.cian.ru' + url_full

            # ID объявления из URL
            id_match = re.search(r'/(\d+)/', url_full)
            if not id_match:
                continue
            listing_id = int(id_match.group(1))

            # Пропускаем дубликаты
            if listing_id in seen_ids:
                continue
            seen_ids.add(listing_id)

            # Заголовок и подзаголовок
            title_elem = card.find(attrs={'data-mark': 'OfferTitle'})
            title = title_elem.get_text(strip=True) if title_elem else ""

            subtitle_elem = card.find(attrs={'data-mark': 'OfferSubtitle'})
            subtitle = subtitle_elem.get_text(strip=True) if subtitle_elem else ""

            data_source = subtitle if subtitle else title

            # Цена
            price_elem = card.find(attrs={'data-mark': 'MainPrice'})
            if not price_elem:
                continue
            price = get_price(price_elem.get_text())
            if not price:
                continue

            # Характеристики
            rooms = get_rooms(data_source)
            total_area = get_area(data_source)
            floor, floors = get_floors(data_source)

            # Цена за м2
            price_per_m2 = None
            if total_area and total_area > 0:
                price_per_m2 = round(price / total_area, 2)

            # Адрес и его части
            address_parts = []
            geo_labels = card.find_all(attrs={'data-name': 'GeoLabel'})
            all_labels = []
            for label in geo_labels:
                text = label.get_text(strip=True)
                all_labels.append(text)
                if text and text != "Москва" and not text.startswith("м."):
                    address_parts.append(text)
            address = ", ".join(address_parts) if address_parts else None

            # Разбираем адрес на компоненты
            ao, district, street, house = parse_address_parts(all_labels)

            # Метро
            metro = None
            metro_minutes = None
            metro_mode = None

            special_geo = card.find(attrs={'data-name': 'SpecialGeo'})
            if special_geo:
                metro_link = special_geo.find('a')
                if metro_link:
                    metro = metro_link.get_text(strip=True)

                remoteness = special_geo.find(class_=lambda c: c and 'remoteness' in c)
                if remoteness:
                    metro_text = remoteness.get_text(strip=True)
                    # Извлекаем минуты
                    mins = re.search(r'(\d+)', metro_text)
                    if mins:
                        metro_minutes = int(mins.group(1))
                    # Тип транспорта
                    if 'пешком' in metro_text.lower():
                        metro_mode = 'walk'
                    elif 'транспорт' in metro_text.lower():
                        metro_mode = 'transport'

            # Описание
            description = None
            desc_elem = card.find(attrs={'data-name': 'Description'})
            if desc_elem:
                description = desc_elem.get_text(strip=True)
                # Убираем переносы строк для CSV
                description = description.replace('\n', ' ').replace('\r', ' ')
                # Убираем множественные пробелы
                description = re.sub(r'\s+', ' ', description).strip()

            # Тип недвижимости
            property_type = get_property_type(data_source)

            # Эксклюзив на Циан
            only_on_cian = False
            card_text = card.get_text(' ', strip=True).lower()
            if 'только на циан' in card_text:
                only_on_cian = True

            # Условия оплаты
            utilities_included = None
            counters_included = None
            deposit_amount = None
            min_term = None
            min_term_months = None
            is_long_term = None
            is_short_term = None

            price_info_elem = card.find(attrs={'data-mark': 'PriceInfo'})
            if price_info_elem:
                price_info_text = price_info_elem.get_text(strip=True)
                (utilities_included, counters_included, commission_amount, deposit_amount,
                 min_term, min_term_months, is_long_term, is_short_term) = parse_payment_info(price_info_text)

            # Сохраняем объявление
            all_listings.append({
                'id': listing_id,
                'url': url_full,
                'title': title,
                'price': price,
                'rooms': rooms,
                'total_area': total_area,
                'floor': floor,
                'floors': floors,
                'address': address,
                'district': district,
                'ao': ao,
                'street': street,
                'house': house,
                'metro': metro,
                'metro_minutes': metro_minutes,
                'metro_mode': metro_mode,
                'description': description,
                'price_per_m2': price_per_m2,
                'property_type': property_type,
                'only_on_cian': only_on_cian,
                'deposit_amount': deposit_amount,
                'utilities_included': utilities_included,
                'counters_included': counters_included,
                'min_term': min_term,
                'min_term_months': min_term_months,
                'is_long_term': is_long_term,
                'is_short_term': is_short_term,
                'scraped_at': scraped_at
            })
            new_count += 1

        print(f"Страница {page}: добавлено {new_count} новых объявлений (всего {len(all_listings)})")

        if new_count == 0:
            break

        time.sleep(1)  # Пауза между страницами

    time.sleep(2)  # Пауза между диапазонами

# Закрываем браузер
driver.quit()

print(f"\nВсего собрано: {len(all_listings)} объявлений")

# Создаем DataFrame
df = pd.DataFrame(all_listings)

# Сохраняем собранные данные
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
filename = f'moscow_rent_monthly_{timestamp}.csv'

df.to_csv(f'data/{filename}', index=False)
print(f"\nДанные сохранены: {filename}")

Браузер запущен
Будем собирать данные по 1 ценовым диапазонам

Диапазон 1/1: 40000 - 50000
Страница 1: добавлено 28 новых объявлений (всего 28)
Страница 1: добавлено 28 новых объявлений (всего 28)

Всего собрано: 28 объявлений

Данные сохранены: moscow_rent_monthly_20251112_112527.csv

Всего собрано: 28 объявлений

Данные сохранены: moscow_rent_monthly_20251112_112527.csv


## 3. Обогащение данных через API

Теперь будем добавлять координаты и информацию об инфраструктуре используя API.

**Используемые API:**
1. **Yandex Geocoder** - преобразование адресов в координаты (23 ключа)
2. **Overpass API** - получение данных об инфраструктуре (магазины, кафе, школы, поликлиники)
3. **HeiGIT OpenRouteService** - расчет времени пешком до центра и метро (17 ключей)

**Примечание:** В реальном проекте данные собирались через HeiGIT API для точного расчета
времени пешком с учетом реальной дорожной сети. Здесь для упрощения и скорости
используем математические формулы, имитируя результаты полученные за долгий промежуток времени.

In [ ]:
df = pd.read_csv('data/slice10.csv')
print(f"Загружено {len(df)} объявлений")

Загружено 50 объявлений


In [ ]:
# API ключи
YANDEX_KEYS = [
    'c27c0648-4a1e-4fa7-a72a-0ccdc81ab450',
    'e7821a87-4fe2-4d8a-873e-7fd5cebaa1f8',
    '1e74be85-3c0e-4e29-a525-80169771c9c1',
    'a995d6b2-e3be-41fe-9f7b-81193a9a5dd8',
    '04796103-1287-4d99-b024-bdf75a7a9efc',
    '0da324d2-5c44-49c6-99c5-3d5c9a613319',
    '1576105b-cb32-4e3d-81d4-0f0ecc361cf2',
    'eef86bab-4247-471b-b45d-2ef7d0432e97',
    '9619c193-d4b6-4fc7-bd7d-b866703d9e5a',
    '1ded743c-0ce5-4c06-a076-a6966c88ed27',
    '746d0a9b-09ed-4ebe-b8f0-7548eaa081b0',
    '65edba0a-527f-4e23-9aa9-7d9543197f13',
    '090ba7e2-5765-435f-a26a-54ee618e767b',
    '6a771be3-b1a9-4a17-9b5d-23d16cac5e6a',
    '9fff59f0-5eab-4a8b-9fb5-6e895db84c16',
    'ddb02436-ec79-4206-a147-c58590b89298',
    '58421de2-c952-41b7-b8b9-d695c2d40dcf',
    'fb895ba2-aac5-4938-84db-12c4e00d1b0a',
    '251177ca-9dcd-4238-84a4-65f4f8c92286',
    '4d4f7ddb-c737-487e-b9a2-7c249c5b8e27',
    'bd52ed89-09d4-496f-95f1-65c4ca02b704',
    '70f82461-7f9e-4ec9-a644-d1a328e6c944',
    '482fb632-b0fe-45b6-9b38-5c3779c47286',
]

# HeiGIT ключи
HEIGIT_KEYS = [
    'eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6ImM4Y2FlZDU3NDkzMTQzZTQ5MjZhNzY5M2NhMjcwZDk1IiwiaCI6Im11cm11cjY0In0=',
    'eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6ImFkMTYyN2FmNjgxYTQ4NmZhOThiOTU0NjEzNGY5MzdlIiwiaCI6Im11cm11cjY0In0=',
    'eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6IjI0MmFkZjdmOTI3YjRiZWVhOWQwODkzYzJjMzJkOGQxIiwiaCI6Im11cm11cjY0In0=',
    'eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6IjdlMDE1OTcxMWE2ZTQ2OWVhZWY2YTAwZWQ2ZWY3Njg3IiwiaCI6Im11cm11cjY0In0=',
    'eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6ImNjODU0NGI0NzQ2NDRhNDk4ZGE3NDcwMDU3ZWVjNDdhIiwiaCI6Im11cm11cjY0In0=',
    'eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6IjllN2NjMGIwMThhMTQyZmM5ZGM3ZjU0NmQ5YTE2YTdjIiwiaCI6Im11cm11cjY0In0=',
    'eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6ImI0Y2M4ODQ0YmJlMjRlNDA5ODcyNGQ2OWE1YmQ4MDFmIiwiaCI6Im11cm11cjY0In0=',
    'eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6IjcxOTJlZmYxNGIwMzQ5MTU4YTI2Y2Y0ZmY5MTI5YmQ1IiwiaCI6Im11cm11cjY0In0=',
    'eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6IjU1ODUyY2Y4YmZiYjRkZGJiOGUzMWM0YzJkMTA5OTRjIiwiaCI6Im11cm11cjY0In0=',
    'eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6IjdmMjhjNDMyYTk1NjRjOTFiOTBlMmIyOGE1NGE3ZGMwIiwiaCI6Im11cm11cjY0In0=',
    'eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6IjBmODc0NjdlNzYxMjQwYTlhMGRlMzFiYjMyODJjZWNjIiwiaCI6Im11cm11cjY0In0=',
    'eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6ImEwYTQxN2JhZmY4NjQ1NTFiNmMyNzFmZGZiZGIwNjRmIiwiaCI6Im11cm11cjY0In0=',
    'eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6IjI3Y2FhOTAwNWI1MzRlMGE5YWMzZmQ2ZDI3MTgxNWY0IiwiaCI6Im11cm11cjY0In0=',
    'eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6ImVhZTBiZWUzNzA2MDRmM2FhZTI0M2Q4ODk0ZWZjN2EwIiwiaCI6Im11cm11cjY0In0=',
    'eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6IjdjM2NlNTg4ZTVhNTRjN2E5ZDRkZGE0MjE0NWFiZDFkIiwiaCI6Im11cm11cjY0In0=',
    'eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6IjMyNzkzNmZmOWU3YjQzNzU5ZDY0MWZkOWJiZWU1NzQzIiwiaCI6Im11cm11cjY0In0=',
    'eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6IjRkYjI2YzgwOTBkMzQzMmViMjI0ZjA4NjIxMGNiODFkIiwiaCI6Im11cm11cjY0In0='
]

# Индексы для ротации ключей (начинаем с первого ключа)
yandex_key_index = 0
heigit_key_index = 0

# Координаты центра Москвы (Красная площадь)
CENTER_LAT = 55.753930
CENTER_LON = 37.620795

# Словарь станций метро с координатами
metro_stations = {
    "Авиамоторная": (55.751831, 37.717848),
    "Автозаводская": (55.706612, 37.657866),
    "Академическая": (55.687835, 37.573215),
    "Александровский сад": (55.752968, 37.608807),
    "Алексеевская": (55.807866, 37.638787),
    "Алма-Атинская": (55.632115, 37.765258),
    "Алтуфьево": (55.897749, 37.587105),
    "Аннино": (55.582722, 37.596572),
    "Арбатская": (55.752212, 37.603311),
    "Аэропорт": (55.800441, 37.532078),
    "Бабушкинская": (55.869322, 37.664208),
    "Багратионовская": (55.743396, 37.497334),
    "Баррикадная": (55.760532, 37.581066),
    "Бауманская": (55.772415, 37.678466),
    "Беговая": (55.773695, 37.546802),
    "Белорусская": (55.776445, 37.583924),
    "Беляево": (55.642424, 37.525858),
    "Бибирево": (55.883965, 37.603735),
    "Библиотека имени Ленина": (55.750291, 37.609857),
    "Битцевский парк": (55.600355, 37.556572),
    "Борисово": (55.632536, 37.743847),
    "Боровицкая": (55.750552, 37.608873),
    "Ботанический сад": (55.845215, 37.638396),
    "Братеево": (55.631535, 37.750985),
    "Братиславская": (55.659695, 37.750908),
    "Бульвар Адмирала Ушакова": (55.545335, 37.542848),
    "Бульвар Дмитрия Донского": (55.569572, 37.577215),
    "Бульвар Рокоссовского": (55.816572, 37.735424),
    "Бунинская аллея": (55.538115, 37.515572),
    "Варшавская": (55.653335, 37.619424),
    "ВДНХ": (55.821572, 37.641466),
    "Владыкино": (55.847572, 37.589424),
    "Водный стадион": (55.839572, 37.487424),
    "Войковская": (55.818572, 37.497424),
    "Волгоградский проспект": (55.725572, 37.687424),
    "Волжская": (55.690572, 37.754424),
    "Волоколамская": (55.835572, 37.382424),
    "Воробьёвы горы": (55.710572, 37.559424),
    "Выставочная": (55.750572, 37.542424),
    "Выхино": (55.715572, 37.817424),
    "Деловой центр": (55.749572, 37.539424),
    "Динамо": (55.789572, 37.558424),
    "Дмитровская": (55.807572, 37.579424),
    "Добрынинская": (55.728572, 37.622424),
    "Домодедовская": (55.610572, 37.718424),
    "Достоевская": (55.781572, 37.613424),
    "Дубровка": (55.718572, 37.679424),
    "Жулебино": (55.684572, 37.854424),
    "Зябликово": (55.611572, 37.744424),
    "Измайловская": (55.787572, 37.781424),
    "Калужская": (55.656572, 37.540424),
    "Кантемировская": (55.635572, 37.656424),
    "Каховская": (55.653572, 37.598424),
    "Каширская": (55.655572, 37.648424),
    "Киевская": (55.743572, 37.567424),
    "Китай-город": (55.755572, 37.635424),
    "Кожуховская": (55.706572, 37.685424),
    "Коломенская": (55.677572, 37.663424),
    "Коммунарка": (55.560572, 37.467424),
    "Комсомольская": (55.775572, 37.656424),
    "Коньково": (55.632572, 37.519424),
    "Красногвардейская": (55.613572, 37.744424),
    "Краснопресненская": (55.760572, 37.577424),
    "Красносельская": (55.780572, 37.667424),
    "Красные Ворота": (55.769572, 37.649424),
    "Крестьянская Застава": (55.732572, 37.667424),
    "Кропоткинская": (55.745572, 37.604424),
    "Крылатское": (55.756572, 37.408424),
    "Кузнецкий Мост": (55.761572, 37.624424),
    "Кузьминки": (55.705572, 37.765424),
    "Кунцевская": (55.731572, 37.446424),
    "Курская": (55.758572, 37.661424),
    "Кутузовская": (55.739572, 37.534424),
    "Ленинский проспект": (55.707572, 37.586424),
    "Лермонтовский проспект": (55.701572, 37.851424),
    "Лефортово": (55.764572, 37.704424),
    "Лихоборы": (55.855572, 37.560424),
    "Ломоносовский проспект": (55.706572, 37.518424),
    "Лубянка": (55.759572, 37.626424),
    "Люблино": (55.675572, 37.761424),
    "Марксистская": (55.741572, 37.656424),
    "Марьина Роща": (55.793572, 37.616424),
    "Марьино": (55.648572, 37.744424),
    "Маяковская": (55.769572, 37.596424),
    "Медведково": (55.887572, 37.661424),
    "Менделеевская": (55.781572, 37.599424),
    "Минская": (55.723572, 37.497424),
    "Митино": (55.845572, 37.361424),
    "Молодёжная": (55.741572, 37.416424),
    "Мякинино": (55.825572, 37.385424),
    "Нагатинская": (55.683572, 37.622424),
    "Нагорная": (55.672572, 37.610424),
    "Нахимовский проспект": (55.662572, 37.605424),
    "Некрасовка": (55.687572, 37.921424),
    "Новогиреево": (55.752572, 37.817424),
    "Новокосино": (55.745572, 37.864424),
    "Новопеределкино": (55.638572, 37.353424),
    "Новослободская": (55.779572, 37.601424),
    "Новоясеневская": (55.601572, 37.553424),
    "Октябрьская": (55.729572, 37.611424),
    "Октябрьское Поле": (55.793572, 37.493424),
    "Орехово": (55.613572, 37.695424),
    "Отрадное": (55.863572, 37.604424),
    "Охотный Ряд": (55.757572, 37.617424),
    "Павелецкая": (55.731572, 37.638424),
    "Парк Победы": (55.736572, 37.517424),
    "Парк культуры": (55.735572, 37.593424),
    "Партизанская": (55.788572, 37.751424),
    "Первомайская": (55.794572, 37.799424),
    "Перово": (55.751572, 37.787424),
    "Петровско-Разумовская": (55.836572, 37.574424),
    "Печатники": (55.692572, 37.728424),
    "Пионерская": (55.736572, 37.467424),
    "Планерная": (55.859572, 37.436424),
    "Площадь Гагарина": (55.706572, 37.586424),
    "Площадь Ильича": (55.746572, 37.681424),
    "Площадь Революции": (55.756572, 37.623424),
    "Полежаевская": (55.777572, 37.519424),
    "Полянка": (55.736572, 37.618424),
    "Пражская": (55.612572, 37.604424),
    "Преображенская площадь": (55.796572, 37.715424),
    "Прокшино": (55.551572, 37.449424),
    "Пролетарская": (55.731572, 37.667424),
    "Проспект Вернадского": (55.677572, 37.506424),
    "Проспект Мира": (55.780572, 37.633424),
    "Профсоюзная": (55.678572, 37.563424),
    "Пушкинская": (55.765572, 37.605424),
    "Пятницкое шоссе": (55.856572, 37.354424),
    "Раменки": (55.697572, 37.499424),
    "Речной вокзал": (55.855572, 37.476424),
    "Рижская": (55.792572, 37.636424),
    "Римская": (55.746572, 37.682424),
    "Румянцево": (55.633572, 37.441424),
    "Рязанский проспект": (55.716572, 37.793424),
    "Савеловская": (55.794572, 37.587424),
    "Саларьево": (55.621572, 37.424424),
    "Свиблово": (55.855572, 37.654424),
    "Севастопольская": (55.651572, 37.598424),
    "Семёновская": (55.782572, 37.719424),
    "Серпуховская": (55.726572, 37.625424),
    "Славянский бульвар": (55.730572, 37.469424),
    "Смоленская": (55.748572, 37.584424),
    "Сокол": (55.805572, 37.515424),
    "Сокольники": (55.789572, 37.679424),
    "Спартак": (55.818572, 37.435424),
    "Спортивная": (55.723572, 37.564424),
    "Сретенский бульвар": (55.766572, 37.635424),
    "Строгино": (55.804572, 37.402424),
    "Студенческая": (55.738572, 37.548424),
    "Сухаревская": (55.772572, 37.632424),
    "Сходненская": (55.850572, 37.439424),
    "Таганская": (55.742572, 37.653424),
    "Тверская": (55.765572, 37.605424),
    "Театральная": (55.758572, 37.618424),
    "Текстильщики": (55.708572, 37.732424),
    "Теплый Стан": (55.618572, 37.507424),
    "Тимирязевская": (55.818572, 37.575424),
    "Третьяковская": (55.740572, 37.627424),
    "Тропарёво": (55.646572, 37.472424),
    "Трубная": (55.767572, 37.622424),
    "Тульская": (55.708572, 37.622424),
    "Тургеневская": (55.765572, 37.636424),
    "Тушинская": (55.826572, 37.436424),
    "Улица 1905 года": (55.764572, 37.561424),
    "Улица Академика Янгеля": (55.595572, 37.601424),
    "Улица Горчакова": (55.542572, 37.531424),
    "Улица Скобелевская": (55.548572, 37.554424),
    "Улица Старокачаловская": (55.569572, 37.576424),
    "Университет": (55.692572, 37.534424),
    "Филатов Луг": (55.601572, 37.407424),
    "Фили": (55.746572, 37.514424),
    "Фонвизинская": (55.822572, 37.588424),
    "Фрунзенская": (55.727572, 37.580424),
    "Царицыно": (55.621572, 37.669424),
    "Цветной бульвар": (55.771572, 37.620424),
    "Черкизовская": (55.804572, 37.745424),
    "Чертановская": (55.640572, 37.606424),
    "Чеховская": (55.765572, 37.608424),
    "Чистые пруды": (55.764572, 37.638424),
    "Чкаловская": (55.755572, 37.659424),
    "Шаболовская": (55.719572, 37.608424),
    "Шелепиха": (55.757572, 37.525424),
    "Шипиловская": (55.621572, 37.743424),
    "Шоссе Энтузиастов": (55.758572, 37.751424),
    "Щёлковская": (55.809572, 37.798424),
    "Щукинская": (55.807572, 37.466424),
    "Электрозаводская": (55.782572, 37.705424),
    "Юго-Западная": (55.664572, 37.483424),
    "Южная": (55.622572, 37.609424),
    "Ясенево": (55.606572, 37.533424)
}

print(f"Готово {len(YANDEX_KEYS)} ключей Яндекс")
print(f"Готово {len(HEIGIT_KEYS)} ключей HeiGIT")
print(f"Загружено {len(metro_stations)} станций метро")

Готово 23 ключей Яндекс
Готово 17 ключей HeiGIT
Загружено 190 станций метро


In [ ]:
# Лимиты для API ключей
YANDEX_DAILY_LIMIT = 1000  # Яндекс геокодер: 25000 запросов в день на ключ
HEIGIT_DAILY_LIMIT = 2000   # HeiGIT OpenRouteService: 20000 запросов в день на ключ

# Счетчики использования ключей
yandex_key_usage = [0] * len(YANDEX_KEYS)
heigit_key_usage = [0] * len(HEIGIT_KEYS)

def get_next_yandex_key():
    """Получает следующий доступный ключ Яндекс с учетом лимитов"""
    global yandex_key_index

    # Проверяем все ключи начиная с текущего
    for _ in range(len(YANDEX_KEYS)):
        # Если ключ не исчерпан - используем его
        if yandex_key_usage[yandex_key_index] < YANDEX_DAILY_LIMIT:
            key = YANDEX_KEYS[yandex_key_index]
            yandex_key_usage[yandex_key_index] += 1
            yandex_key_index = (yandex_key_index + 1) % len(YANDEX_KEYS)
            return key
        # Иначе переходим к следующему
        yandex_key_index = (yandex_key_index + 1) % len(YANDEX_KEYS)

    # Все ключи исчерпаны
    print("ВНИМАНИЕ: Все Yandex ключи исчерпали дневной лимит!")
    return None

def get_next_heigit_key():
    """Получает следующий доступный ключ HeiGIT с учетом лимитов"""
    global heigit_key_index

    # Проверяем все ключи начиная с текущего
    for _ in range(len(HEIGIT_KEYS)):
        # Если ключ не исчерпан - используем его
        if heigit_key_usage[heigit_key_index] < HEIGIT_DAILY_LIMIT:
            key = HEIGIT_KEYS[heigit_key_index]
            heigit_key_usage[heigit_key_index] += 1
            heigit_key_index = (heigit_key_index + 1) % len(HEIGIT_KEYS)
            return key
        # Иначе переходим к следующему
        heigit_key_index = (heigit_key_index + 1) % len(HEIGIT_KEYS)

    # Все ключи исчерпаны
    print("ВНИМАНИЕ: Все HeiGIT ключи исчерпали дневной лимит!")
    return None

print("Система отслеживания лимитов готова")
print(f"Максимальная емкость в день:")
print(f"  Яндекс: {len(YANDEX_KEYS)} ключей × {YANDEX_DAILY_LIMIT:,} = {len(YANDEX_KEYS) * YANDEX_DAILY_LIMIT:,} запросов")
print(f"  HeiGIT: {len(HEIGIT_KEYS)} ключей × {HEIGIT_DAILY_LIMIT:,} = {len(HEIGIT_KEYS) * HEIGIT_DAILY_LIMIT:,} запросов")

Система отслеживания лимитов готова
Максимальная емкость в день:
  Яндекс: 23 ключей × 1,000 = 23,000 запросов
  HeiGIT: 17 ключей × 2,000 = 34,000 запросов


In [ ]:
# Функция для получения координат по адресу (Yandex Geocoder API)
def get_coordinates(address):
    # Получаем ключ с учетом лимитов
    api_key = get_next_yandex_key()
    if not api_key:
        return None, None

    url = "https://geocode-maps.yandex.ru/1.x/"
    params = {
        "apikey": api_key,
        "geocode": address,
        "format": "json",
        "lang": "ru_RU"
    }

    response = requests.get(url, params=params, timeout=10)
    data = response.json()

    members = data["response"]["GeoObjectCollection"]["featureMember"]
    if not members:
        return None, None

    pos = members[0]["GeoObject"]["Point"]["pos"]
    lon, lat = map(float, pos.split())

    # Проверка что координаты в Москве
    if not (55.0 <= lat <= 56.5 and 36.5 <= lon <= 38.2):
        return None, None

    return lat, lon

print("Функция геокодирования готова")

Функция геокодирования готова


In [ ]:
def haversine_distance(lat1, lon1, lat2, lon2):
    """Вычисляет расстояние по прямой между двумя точками в километрах"""
    # Радиус Земли в км
    R = 6371

    # Переводим в радианы
    lat1_rad = math.radians(lat1)
    lat2_rad = math.radians(lat2)
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)

    # Формула гаверсинуса
    a = math.sin(dlat/2)**2 + math.cos(lat1_rad) * math.cos(lat2_rad) * math.sin(dlon/2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))

    return R * c

def get_walking_time_heigit(lat1, lon1, lat2, lon2):
    """Получает время в пути пешком через HeiGIT API (используется для демонстрации)

    ПРИМЕЧАНИЕ: В реальном сборе данных мы использовали HeiGIT OpenRouteService API
    для точного расчета времени пешком с учетом реальной дорожной сети.
    Здесь для упрощения и скорости используем математическую формулу,
    имитируя результаты которые были получены за долгий промежуток времени.
    """
    # Получаем ключ с учетом лимитов
    api_key = get_next_heigit_key()
    if not api_key:
        return None

    url = 'https://api.openrouteservice.org/v2/directions/foot-walking'
    body = {'coordinates': [[lon1, lat1], [lon2, lat2]]}
    headers = {'Authorization': api_key, 'Content-Type': 'application/json'}

    response = requests.post(url, json=body, headers=headers, timeout=10)
    data = response.json()
    duration_sec = data['routes'][0]['summary']['duration']

    return duration_sec

print("Функции расчета расстояния готовы")

Функции расчета расстояния готовы


In [ ]:
# Функция для получения информации об инфраструктуре (Overpass API)
def get_infrastructure(lat, lon, radius=500):
    """Получает количество объектов инфраструктуры через Overpass API

    Используем несколько зеркал Overpass API для балансировки нагрузки
    """
    # Список зеркал Overpass API
    overpass_urls = [
        "https://overpass-api.de/api/interpreter",
        "https://z.overpass-api.de/api/interpreter",
        "https://overpass.kumi.systems/api/interpreter"
    ]

    # Формируем запрос для Overpass
    query = f"""
    [out:json][timeout:25];
    (
      node(around:{radius},{lat},{lon})["shop"];
      node(around:{radius},{lat},{lon})["amenity"~"^(cafe|restaurant|fast_food)$"];
      node(around:{radius},{lat},{lon})["amenity"~"^(school|college|university|kindergarten)$"];
      node(around:{radius},{lat},{lon})["amenity"~"^(hospital|clinic|pharmacy|dentist)$"];
    );
    out tags;
    """

    # Пробуем разные зеркала
    for url in overpass_urls:
        try:
            response = requests.post(url, data={'data': query}, timeout=30)

            # Проверяем статус ответа
            if response.status_code == 200:
                data = response.json()
                break
            elif response.status_code == 429:
                # Слишком много запросов, пробуем следующее зеркало
                print(f"  Overpass API {url}: лимит превышен, пробуем следующее зеркало...")
                continue
            else:
                # Другая ошибка
                print(f"  Overpass API {url}: ошибка {response.status_code}")
                continue

        except requests.exceptions.Timeout:
            print(f"  Overpass API {url}: таймаут")
            continue
        except requests.exceptions.JSONDecodeError:
            print(f"  Overpass API {url}: неверный формат ответа")
            continue
        except Exception as e:
            print(f"  Overpass API {url}: ошибка {e}")
            continue
    else:
        # Все зеркала не сработали - возвращаем нули
        print(f"  Overpass API: все зеркала недоступны, возвращаем нули")
        return 0, 0, 0, 0

    # Считаем объекты по категориям
    shops = 0
    cafes = 0
    education = 0
    healthcare = 0

    for element in data.get('elements', []):
        tags = element.get('tags', {})

        if 'shop' in tags:
            shops += 1

        amenity = tags.get('amenity')
        if amenity in ['cafe', 'restaurant', 'fast_food']:
            cafes += 1
        elif amenity in ['school', 'college', 'university', 'kindergarten']:
            education += 1
        elif amenity in ['hospital', 'clinic', 'pharmacy', 'dentist']:
            healthcare += 1

    return shops, cafes, education, healthcare

print("Функция получения инфраструктуры готова")

Функция получения инфраструктуры готова


In [ ]:
# Очищаем название станции метро
def clean_metro_name(name):
    if not name or (isinstance(name, float) and pd.isna(name)):
        return None
    name = str(name)
    # Убираем "м." и "метро"
    name = re.sub(r'^\s*(м\.|метро)\s+', '', name, flags=re.IGNORECASE)
    # Убираем скобки
    name = re.sub(r'\(.*?\)', '', name)
    # Заменяем тире
    name = name.replace('—', '-').replace('–', '-')
    # Убираем лишние пробелы
    name = re.sub(r'\s+', ' ', name).strip()
    return name

In [ ]:
# Словарь для кэширования координат по адресам
address_coords = {}

# Словарь для кэширования инфраструктуры по координатам (округленным)
infra_cache = {}

print("Кэш готов")

Кэш готов


In [ ]:
# ВАЖНО: Этот код обрабатывает все объявления, может занять несколько часов!
# Для теста можно взять первые 100 строк: df_sample = df.head(100)

enriched_data = []
failed_data = []

print(f"Начинаем обогащение {len(df)} объявлений...\n")

for idx, row in df.iterrows():
    if (idx + 1) % 100 == 0:
        print(f"Обработано {idx + 1}/{len(df)}...")

    result = {
        'listing_id': row['id'],
        'address': row['address']
    }

    # Пропускаем если нет адреса
    if pd.isna(row['address']) or not row['address']:
        result['error'] = 'NO_ADDRESS'
        failed_data.append(result)
        continue

    address = str(row['address']).strip().lower()

    # Шаг 1: Получаем координаты
    if address in address_coords:
        # Берем из кэша
        lat, lon = address_coords[address]
        geocode_reused = True
    else:
        # Запрашиваем у API
        lat, lon = get_coordinates("Москва, " + row['address'])

        if lat is None:
            result['error'] = 'GEOCODE_FAILED'
            failed_data.append(result)
            continue

        address_coords[address] = (lat, lon)
        geocode_reused = False
        time.sleep(0.05)  # Небольшая пауза

    result['lat'] = round(lat, 6)
    result['lon'] = round(lon, 6)
    result['geocode_reused'] = geocode_reused

    # Шаг 2: Расстояние и время до центра
    center_distance_km = haversine_distance(lat, lon, CENTER_LAT, CENTER_LON)
    result['center_distance_km'] = round(center_distance_km, 2)

    # Вычисляем время пешком: расстояние в метрах / скорость 1.25 м/с * коэффициент 1.15 (извилистость дорог)
    center_walk_sec = int((center_distance_km * 1000 / 1.25) * 1.15)
    result['center_walk_sec'] = center_walk_sec
    result['center_source'] = 'estimated'  # Указываем что данные расчетные

    # Оценка доступности центра (экспоненциальный спад с полураспадом 30 мин)
    center_score = math.exp(-center_walk_sec / 1800)
    result['center_score'] = round(center_score, 4)

    # Шаг 3: Информация о метро
    metro_name = clean_metro_name(row.get('metro'))
    metro_walk_sec = None
    metro_distance_km = None
    metro_lat = None
    metro_lon = None
    metro_source = None

    if metro_name and metro_name in metro_stations:
        # Станция найдена в справочнике
        metro_lat, metro_lon = metro_stations[metro_name]

        # Если есть время из Циан - используем его
        if pd.notna(row.get('metro_minutes')):
            metro_walk_sec = int(row['metro_minutes'] * 60)
            # Рассчитываем расстояние исходя из времени: скорость пешком ~4.8 км/ч = 0.08 км/мин
            metro_distance_km = (metro_walk_sec / 60) * 0.08
            metro_source = 'cian'
        else:
            # Если времени нет - вычисляем по формуле от расстояния
            metro_distance_km = haversine_distance(lat, lon, metro_lat, metro_lon)
            metro_walk_sec = int((metro_distance_km * 1000) / 1.25)
            metro_source = 'computed'
    else:
        # Ищем ближайшую станцию метро
        min_dist = float('inf')
        closest_station = None
        closest_lat = None
        closest_lon = None

        for station_name, (station_lat, station_lon) in metro_stations.items():
            dist = haversine_distance(lat, lon, station_lat, station_lon)
            if dist < min_dist:
                min_dist = dist
                closest_station = station_name
                closest_lat = station_lat
                closest_lon = station_lon

        if closest_station:
            metro_name = closest_station
            metro_lat = closest_lat
            metro_lon = closest_lon
            metro_distance_km = min_dist
            metro_walk_sec = int((min_dist * 1000) / 1.25)
            metro_source = 'computed'

    result['metro_name'] = metro_name
    result['metro_lat'] = metro_lat
    result['metro_lon'] = metro_lon
    result['metro_walk_sec'] = metro_walk_sec
    result['metro_distance_km'] = round(metro_distance_km, 2) if metro_distance_km else None
    result['metro_source'] = metro_source

    # Оценка доступности метро
    if metro_walk_sec:
        metro_score = math.exp(-metro_walk_sec / 1200)
    else:
        metro_score = 0.0
    result['metro_score'] = round(metro_score, 4)

    # Шаг 4: Инфраструктура
    # Округляем координаты для кэша (тайлы ~330м)
    tile_lat = round(round(lat / 0.003) * 0.003, 6)
    tile_lon = round(round(lon / 0.003) * 0.003, 6)
    tile_key = f"{tile_lat}:{tile_lon}"

    if tile_key in infra_cache:
        shops, cafes, education, healthcare = infra_cache[tile_key]
    else:
        shops, cafes, education, healthcare = get_infrastructure(lat, lon)
        infra_cache[tile_key] = (shops, cafes, education, healthcare)
        time.sleep(0.5)  # Пауза для Overpass API

    infra_total = shops + cafes + education + healthcare

    result['infra_shops'] = shops
    result['infra_cafes'] = cafes
    result['infra_education'] = education
    result['infra_healthcare'] = healthcare
    result['infra_total'] = infra_total

    # Оценка инфраструктуры
    infra_score = min(1.0, infra_total / 60)
    result['infra_score'] = round(infra_score, 4)

    # Шаг 5: Итоговая метрика доступности
    accessibility_index = 0.3 * center_score + 0.4 * metro_score + 0.3 * infra_score
    result['accessibility_index'] = round(accessibility_index, 3)

    enriched_data.append(result)

    # Сохраняем промежуточные результаты каждые 500 записей
    if (idx + 1) % 500 == 0:
        pd.DataFrame(enriched_data).to_csv('data/processed/checkpoint.csv', index=False)
        print(f"Чекпоинт сохранен: {len(enriched_data)} успешных")

print(f"\nГотово! Успешно: {len(enriched_data)}, ошибок: {len(failed_data)}")

# Выводим статистику использования API ключей
print(f"\n=== Статистика использования API ключей ===")
print(f"\nYandex Geocoder:")
total_yandex = sum(yandex_key_usage)
print(f"  Всего запросов: {total_yandex:,}")
print(f"  Использовано ключей: {sum(1 for u in yandex_key_usage if u > 0)} из {len(YANDEX_KEYS)}")
exhausted_yandex = sum(1 for u in yandex_key_usage if u >= YANDEX_DAILY_LIMIT)
if exhausted_yandex > 0:
    print(f"  Исчерпанных ключей: {exhausted_yandex}")
print(f"  Средняя нагрузка на ключ: {total_yandex / len(YANDEX_KEYS):.0f} запросов")

print(f"\nHeiGIT OpenRouteService:")
total_heigit = sum(heigit_key_usage)
print(f"  Всего запросов: {total_heigit:,}")
print(f"  Использовано ключей: {sum(1 for u in heigit_key_usage if u > 0)} из {len(HEIGIT_KEYS)}")
exhausted_heigit = sum(1 for u in heigit_key_usage if u >= HEIGIT_DAILY_LIMIT)
if exhausted_heigit > 0:
    print(f"  Исчерпанных ключей: {exhausted_heigit}")
print(f"  Средняя нагрузка на ключ: {total_heigit / len(HEIGIT_KEYS):.0f} запросов")

Начинаем обогащение 50 объявлений...

  Overpass API https://overpass-api.de/api/interpreter: ошибка 504
  Overpass API https://overpass-api.de/api/interpreter: ошибка 504
  Overpass API https://overpass-api.de/api/interpreter: ошибка 504
  Overpass API https://overpass-api.de/api/interpreter: ошибка 504
  Overpass API https://overpass-api.de/api/interpreter: ошибка 504
  Overpass API https://overpass-api.de/api/interpreter: ошибка 504


KeyboardInterrupt: 

In [ ]:
# Создаем датафрейм с обогащенными данными
df_enriched = pd.DataFrame(enriched_data)

print(f"Обогащенный датасет: {len(df_enriched)} строк")
df_enriched.head()

Обогащенный датасет: 41 строк


,listing_id,address,lat,lon,geocode_reused,center_distance_km,center_walk_sec,center_source,center_score,metro_name,...,metro_distance_km,metro_source,metro_score,infra_shops,infra_cafes,infra_education,infra_healthcare,infra_total,infra_score,accessibility_index
0,322790784,"САО, р-н Тимирязевский, Дмитровское шоссе, 13А",55.818715,37.573501,False,7.79,7164,estimated,0.0187,Тимирязевская,...,0.08,cian,0.9512,72,14,1,15,102,1.0000,0.686
1,322638146,"САО, р-н Хорошевский, Ленинградский проспект, ...",55.798171,37.534757,False,7.29,6707,estimated,0.0241,Аэропорт,...,0.24,cian,0.8607,121,38,0,20,179,1.0000,0.652
2,323099850,"САО, р-н Ховрино, улица Дыбенко, 22К3",55.872602,37.486652,False,15.63,14381,estimated,0.0003,Речной вокзал,...,2.00,computed,0.2640,45,1,0,10,56,0.9333,0.386
3,323530828,"САО, р-н Аэропорт, Новый Зыковский проезд, 5",55.799310,37.553531,False,6.57,6043,estimated,0.0348,Динамо,...,1.20,cian,0.4724,59,7,1,13,80,1.0000,0.499
4,323549971,"ЮЗАО, р-н Академический, улица Дмитрия Ульянов...",55.689466,37.569971,False,7.84,7215,estimated,0.0182,Академическая,...,0.32,cian,0.8187,122,27,0,17,166,1.0000,0.633


In [ ]:
# Объединяем с исходными данными
df_final = df.merge(df_enriched, left_on='id', right_on='listing_id', how='left')

print(f"Итоговый датасет: {len(df_final)} строк")
print(f"Колонок: {len(df_final.columns)}")

df_final.to_csv('data/processed/moscow_rent_final.csv', index=False)

Итоговый датасет: 50 строк
Колонок: 72


## 4. Анализ данных (EDA)

Исследуем полученный датасет